# Projet Simulation Exponential Backoff

Neil Bonnard 22505907

In [4]:
import random as rd
import seaborn as sns
import numpy as np
from heapq import heappush, heappop

def exp_lambda(λ):
    return np.random.exponential(1 / λ)   #Fonction pour generer temps random selon la loi exponentielle de paramètre λ

def backoff(i, tau):
    return np.random.exponential((2**i) * tau)    #Fonction pour generer temps random selon la loi exponentielle de paramètre (2^i)*ta

#for i in range(10):
#    print(backoff(i, 1))

In [5]:
def simulate(N, lam, K, tau, T_max):

    echeancier = []
    t = 0
    stations = []
    reussis = []
    canal_libre = True
    log = []

    for i in range(N):      #Initialisation des stations a l'état 1 et pas de paquets dans la file d'attente
        stations.append({"queue_len": 0,
            "state" : 1,
            "attempt_scheduled" : False,
            "is_attempting" : False,
            "end_valid": False})
    
    for i in range(N):      #Initialisation de l'echeancier avec les evenements d'arrivee des paquets a chaque station
        t_arrival = exp_lambda(lam)
        heappush(echeancier, (t_arrival, "ARRIVAL", i))


    while t < T_max: 

        evt = heappop(echeancier)     #Recuperation de l'evenement le plus proche
        t = evt[0]
        station = evt[2]
        match evt[1]:
            case "ARRIVAL":   #Si c'est un evenement d'arrivee de paquet
                t_arrival = t + exp_lambda(lam)
                heappush(echeancier, (t_arrival, "ARRIVAL", station))

                if stations[station]["queue_len"] < K:
                    stations[station]["queue_len"] += 1

                if stations[station]["queue_len"] == 1 and not stations[station]["attempt_scheduled"] and not stations[station]["is_attempting"]:
                    heappush(echeancier, (t, "ATTEMPT", station))
                    stations[station]["attempt_scheduled"] = True
                

            case "ATTEMPT":   #Si c'est un la station essaye d'envoyer un paquet

                if canal_libre:
                    heappush(echeancier, (t+1, "END_TX", station))
                    canal_libre = False
                    stations[station]["is_attempting"] = True
                    stations[station]["end_valid"] = True
                    heappush(log, (t, station, "ATTEMPT"))

                else: 
                    stations[station]["is_attempting"] = False
                    stations[station]["attempt_scheduled"] = True
                    stations[station]["state"] += 1
                    t_backoff = t + backoff(stations[station]["state"], tau)
                    heappush(echeancier, (t_backoff, "ATTEMPT", station))
                    for i in range(N):
                        if i != station and stations[i]["is_attempting"]:
                            r = i
                            stations[i]["is_attempting"] = False
                            stations[i]["attempt_scheduled"] = True
                            stations[i]["state"] += 1
                            t_backoff_i = t + backoff(stations[i]["state"], tau)
                            heappush(echeancier, (t_backoff_i, "ATTEMPT", i))
                            stations[i]["end_valid"] = False

                    heappush(log, (t, station, "COLLISION", r))   #On ajoute l'evenement de collision au log
                    canal_libre = True


            case "END_TX":    #Si un envoi de paquet est terminer
                if stations[station]["end_valid"]:   
                    stations[station]["state"] = 1  #On remet la station a l'etat de base
                    stations[station]["queue_len"] -= 1
                    reussis.append((station, t))   #On ajoute le paquet a la liste des paquets reussis avec le temps d'arrivee du paquet
                    canal_libre = True
                    stations[station]["is_attempting"] = False
                    stations[station]["attempt_scheduled"] = False
                    if stations[station]["queue_len"] > 0:   #Si il y a encore des paquets dans la file d'attente de la station, on programme un nouvel attempt
                        heappush(echeancier, (t, "ATTEMPT", station))
                        stations[station]["attempt_scheduled"] = True
                    heappush(log, (t, station, "END_TX"))

    return log

In [8]:
#Parametres
N = 5   #Le nombre de stations 
lam = 0.4   #Parametre de la loi exponentiel pour le temps d'attente entre l'arrivée de deux paquets a une station
K = 5   #Capacité de la file d'attente d'une station
tau = 1  #Parametre de la loi exponentielle pour la durée du Backoff
T_max = 1000  #Durée de la simulation
data = simulate(N, lam, K, tau, T_max)
#print (data)
for i in data:
    if i[2] == "END_TX":
        print (i)

(2.285654875630456, 4, 'END_TX')
(3.9688418692894984, 0, 'END_TX')
(4.9688418692894984, 0, 'END_TX')
(11.154411111507159, 0, 'END_TX')
(12.165667109052045, 0, 'END_TX')
(13.165667109052045, 0, 'END_TX')
(14.165667109052045, 0, 'END_TX')
(16.683718757333125, 0, 'END_TX')
(17.683718757333125, 0, 'END_TX')
(18.683718757333125, 0, 'END_TX')
(19.683718757333125, 0, 'END_TX')
(22.223445824937443, 0, 'END_TX')
(23.223445824937443, 0, 'END_TX')
(24.223445824937443, 0, 'END_TX')
(25.223445824937443, 0, 'END_TX')
(33.272568116527054, 0, 'END_TX')
(34.272568116527054, 0, 'END_TX')
(35.272568116527054, 0, 'END_TX')
(36.272568116527054, 0, 'END_TX')
(37.272568116527054, 0, 'END_TX')
(40.44215472892593, 3, 'END_TX')
(41.44215472892593, 3, 'END_TX')
(42.44215472892593, 3, 'END_TX')
(43.44215472892593, 3, 'END_TX')
(49.9027347187922, 2, 'END_TX')
(50.9027347187922, 2, 'END_TX')
(60.67956205110087, 2, 'END_TX')
(61.67956205110087, 2, 'END_TX')
(62.67956205110087, 2, 'END_TX')
(63.67956205110087, 2, 'EN